In [1]:
from pathlib import Path
import json, time, io, glob, calendar, re
from datetime import datetime, date
import requests
import pandas as pd
from dateutil.relativedelta import relativedelta

ROOT = Path("..").resolve()
RAW = ROOT / "raw"
INTERIM = ROOT / "interim"
CLEAN = ROOT / "clean"
LOGS = ROOT / "logs"

for p in (RAW, INTERIM, CLEAN, LOGS):
    p.mkdir(parents=True, exist_ok=True)
    print(p, "exists:", p.exists())

TODAY = date.today().isoformat()
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:156.0) Gecko/20100101 Firefox/156.0"}
print("collection date:", TODAY)

/home/da25b048/Documents/madras/da2402 DCV/dcvproject/raw exists: True
/home/da25b048/Documents/madras/da2402 DCV/dcvproject/interim exists: True
/home/da25b048/Documents/madras/da2402 DCV/dcvproject/clean exists: True
/home/da25b048/Documents/madras/da2402 DCV/dcvproject/logs exists: True
collection date: 2026-10-24


In [2]:
from amfipy import AMFIClient
client = AMFIClient()

In [3]:
NAV_URL = "https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx"
params = {"frmdt": "01-Sep-2026", "todt": "03-Sep-2026"}

r = requests.get(NAV_URL, params=params, headers=HEADERS, timeout=60)
print("status:", r.status_code)
print("content-type:", r.headers.get("Content-Type"))
print("length:", len(r.text))
print("---- first 300 chars ----")
print(r.text[:300])

status: 200
content-type: text/plain
length: 3477867
---- first 300 chars ----
Scheme Code;NAV Name;Plan;Option;ISIN Div Payout/ISIN Growth;ISIN Div Reinvestment;Net Asset Value;Date

Open Ended Schemes ( Money Market )


Taurus Mutual Fund
139619;Taurus Investor Education Pool - Unclaimed Dividend - Growth;;;;;10.0000;01-Sep-2026
139619;Taurus Investor Education Pool -


In [4]:
nav_probe_file = RAW / f"amfi_nav_probe_{TODAY}.txt"
nav_probe_file.write_text(r.text, encoding="utf-8")

manifest = pd.DataFrame([{
    "file_name": nav_probe_file.name,
    "source_url": r.url,
    "collected_at": datetime.now().isoformat(timespec="seconds"),
    "http_status": r.status_code,
    "bytes": len(r.content),
    "note": "NAV probe: 01-Sep-2026 to 03-Sep-2026",
}])
manifest.to_csv(RAW / "manifest.csv", index=False)
print(manifest.to_string())

                       file_name                                                                                        source_url         collected_at  http_status    bytes                                   note
0  amfi_nav_probe_2026-10-24.txt  https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx?frmdt=01-Sep-2026&todt=03-Sep-2026  2026-10-24T19:36:24          200  3477867  NAV probe: 01-Sep-2026 to 03-Sep-2026


In [5]:
nav_probe = pd.read_csv(nav_probe_file, sep=";")
print("shape:", nav_probe.shape)
print("dtypes:\n", nav_probe.dtypes)
print("columns:", nav_probe.columns.tolist())
print(nav_probe.head(3).to_string())

real = nav_probe.dropna(subset=["Net Asset Value"]).copy()
print("\nreal rows:", len(real), "of", len(nav_probe))
print("\nPlan value_counts:\n", real["Plan"].value_counts(dropna=False))
print("\nOption distinct values:", real["Option"].nunique())
print("\nScheme Code unique:", nav_probe["Scheme Code"].nunique())
print("rows-per-code distribution:")
print(real.groupby("Scheme Code").size().describe())

shape: (27309, 8)
dtypes:
 Scheme Code                        str
NAV Name                           str
Plan                               str
Option                             str
ISIN Div Payout/ISIN Growth        str
ISIN Div Reinvestment              str
Net Asset Value                float64
Date                               str
dtype: object
columns: ['Scheme Code', 'NAV Name', 'Plan', 'Option', 'ISIN Div Payout/ISIN Growth', 'ISIN Div Reinvestment', 'Net Asset Value', 'Date']
                           Scheme Code                                                      NAV Name Plan Option ISIN Div Payout/ISIN Growth ISIN Div Reinvestment  Net Asset Value         Date
0  Open Ended Schemes ( Money Market )                                                           NaN  NaN    NaN                         NaN                   NaN              NaN          NaN
1                   Taurus Mutual Fund                                                           NaN  NaN    NaN           

In [6]:
excel_bytes = client.ter.download_excel(month="08-2026")
print("bytes:", len(excel_bytes))

ter_probe = pd.read_excel(io.BytesIO(excel_bytes))
print("shape:", ter_probe.shape)
print("columns:", ter_probe.columns.tolist())
print("unique NSDL Scheme Code:", ter_probe["NSDL Scheme Code"].nunique())
print("TER Date min/max:", ter_probe["TER Date"].min(), "->", ter_probe["TER Date"].max())

lc_mask = ter_probe["Scheme Category"].str.contains("Large Cap", case=False, na=False)
print("\nLarge Cap rows:", lc_mask.sum())
print("Large Cap unique schemes:", ter_probe.loc[lc_mask, "NSDL Scheme Code"].nunique())
print("\nlarge-cap label variants:")
print(ter_probe.loc[lc_mask, "Scheme Category"].value_counts().to_string())

# save deduped probe as CSV
ter_probe_file = RAW / f"amfi_ter_probe_2026-08_{TODAY}.csv"
(ter_probe.sort_values("TER Date")
          .drop_duplicates(subset=["NSDL Scheme Code"], keep="first")
          .to_csv(ter_probe_file, index=False))

manifest = pd.read_csv(RAW / "manifest.csv")
manifest = pd.concat([manifest, pd.DataFrame([{
    "file_name": ter_probe_file.name,
    "source_url": "amfipy client.ter.download_excel(month=08-2026) -> csv",
    "collected_at": datetime.now().isoformat(timespec="seconds"),
    "http_status": 200,
    "bytes": ter_probe_file.stat().st_size,
    "note": f"TER probe: 2026-08, {ter_probe_file.stat().st_size:,} bytes",
}])], ignore_index=True)
manifest.to_csv(RAW / "manifest.csv", index=False)
print("\nmanifest now has", len(manifest), "rows")

bytes: 4475974
shape: (63699, 15)
columns: ['NSDL Scheme Code', 'Scheme Name', 'Scheme Type', 'Scheme Category', 'TER Date', 'Regular Plan - Base Expense Ratio (BER) (%)', 'Regular Plan - Brokerage cost (%)', 'Regular Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Regular Plan - Statutory Levies (including GST) (%)', 'Regular Plan - Total TER (%)', 'Direct Plan - Base Expense Ratio (BER) (%)', 'Direct Plan - Brokerage cost (%)', 'Direct Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Direct Plan - Statutory Levies (including GST) (%)', 'Direct Plan - Total TER (%)']
unique NSDL Scheme Code: 2150
TER Date min/max: 2026-08-01 00:00:00 -> 2026-08-31 00:00:00

Large Cap rows: 1055
Large Cap unique schemes: 35

large-cap label variants:
Scheme Category
Equity Scheme - Large Cap Fund     724
Equity Schemes - Large Cap Fund    331

manifest now has 2 rows


In [7]:
START = date(2023, 1, 1)
END   = date(2026, 9, 30)

def month_list(start, end):
    cur = date(start.year, start.month, 1)
    while cur <= end:
        yield cur.year, cur.month
        cur = cur + relativedelta(months=1)

months = list(month_list(START, END))
print("months:", len(months), "|", months[0], "to", months[-1])

def get_with_retry(url, params, headers, timeout=120, tries=3, sleep=2.0):
    for k in range(tries):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=timeout)
            if resp.status_code == 200:
                return resp
            time.sleep(sleep * (k + 1))
        except requests.exceptions.RequestException:
            time.sleep(sleep * (k + 1))
    return None

months: 45 | (2023, 1) to (2026, 9)


In [8]:
nav_manifest = []

for year, month in months:
    last_day = calendar.monthrange(year, month)[1]
    frm = date(year, month, 1).strftime("%d-%b-%Y")
    to  = date(year, month, last_day).strftime("%d-%b-%Y")
    fname = RAW / f"amfi_nav_{year}-{month:02d}.txt"

    if fname.exists():
        print(f"skip {fname.name} ({fname.stat().st_size:,} bytes)")
        nav_manifest.append({"file_name": fname.name, "source_url": NAV_URL,
                             "collected_at": datetime.fromtimestamp(fname.stat().st_mtime).isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"NAV {frm} to {to} (cached)"})
        continue

    resp = get_with_retry(NAV_URL, {"frmdt": frm, "todt": to}, HEADERS)
    if resp is None:
        print(f"FAILED {year}-{month:02d}")
        nav_manifest.append({"file_name": None, "source_url": NAV_URL,
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": "FAILED", "bytes": 0, "note": f"NAV {frm} to {to}"})
        continue

    fname.write_text(resp.text, encoding="utf-8")
    print(f"wrote {fname.name}  {len(resp.content):>9,} bytes")
    nav_manifest.append({"file_name": fname.name, "source_url": resp.url,
                         "collected_at": datetime.now().isoformat(timespec="seconds"),
                         "http_status": resp.status_code, "bytes": len(resp.content),
                         "note": f"NAV {frm} to {to}"})
    time.sleep(1.5)

pd.DataFrame(nav_manifest).to_csv(RAW / "manifest_nav.csv", index=False)
print("\nNAV written:", sum(1 for x in nav_manifest if x["file_name"]),
      "| failed:", sum(1 for x in nav_manifest if x["file_name"] is None))

skip amfi_nav_2023-01.txt (220,500 bytes)
skip amfi_nav_2023-02.txt (220,500 bytes)
skip amfi_nav_2023-03.txt (220,500 bytes)
skip amfi_nav_2023-04.txt (17,946,652 bytes)
skip amfi_nav_2023-05.txt (21,840,394 bytes)
skip amfi_nav_2023-06.txt (21,214,378 bytes)
skip amfi_nav_2023-07.txt (21,325,073 bytes)
skip amfi_nav_2023-08.txt (21,779,987 bytes)
skip amfi_nav_2023-09.txt (19,570,974 bytes)
skip amfi_nav_2023-10.txt (20,500,929 bytes)
skip amfi_nav_2023-11.txt (20,438,258 bytes)
skip amfi_nav_2023-12.txt (20,637,123 bytes)
skip amfi_nav_2024-01.txt (21,608,899 bytes)
skip amfi_nav_2024-02.txt (21,174,978 bytes)
skip amfi_nav_2024-03.txt (20,155,232 bytes)
skip amfi_nav_2024-04.txt (20,224,513 bytes)
skip amfi_nav_2024-05.txt (21,718,354 bytes)
skip amfi_nav_2024-06.txt (20,499,460 bytes)
skip amfi_nav_2024-07.txt (23,160,878 bytes)
skip amfi_nav_2024-08.txt (22,525,064 bytes)
skip amfi_nav_2024-09.txt (21,997,071 bytes)
skip amfi_nav_2024-10.txt (23,628,089 bytes)
skip amfi_nav_2024-

In [9]:
ter_manifest = []

for year, month in months:
    month_str = f"{month:02d}-{year}"
    fname = RAW / f"amfi_ter_{year}-{month:02d}.csv"

    if fname.exists():
        print(f"skip {fname.name} ({fname.stat().st_size:,} bytes)")
        ter_manifest.append({"file_name": fname.name,
                             "source_url": "amfipy client.ter.download_excel",
                             "collected_at": datetime.fromtimestamp(fname.stat().st_mtime).isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"TER {month_str} (cached)"})
        continue

    try:
        xb = client.ter.download_excel(month=month_str)
        df_m = pd.read_excel(io.BytesIO(xb))
        df_m = df_m.sort_values("TER Date").drop_duplicates(
            subset=["NSDL Scheme Code"], keep="first")
        df_m.to_csv(fname, index=False)
        print(f"wrote {fname.name}  {len(df_m):>5,} schemes  {fname.stat().st_size:>8,} bytes")
        ter_manifest.append({"file_name": fname.name,
                             "source_url": "amfipy client.ter.download_excel",
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": 200, "bytes": fname.stat().st_size,
                             "note": f"TER {month_str}, {len(df_m)} unique schemes"})
    except Exception as e:
        print(f"FAILED {month_str}: {type(e).__name__}: {e}")
        ter_manifest.append({"file_name": None, "source_url": "amfipy",
                             "collected_at": datetime.now().isoformat(timespec="seconds"),
                             "http_status": "FAILED", "bytes": 0, "note": f"TER {month_str}: {e}"})

    time.sleep(1.5)

pd.DataFrame(ter_manifest).to_csv(RAW / "manifest_ter.csv", index=False)
print("\nTER written:", sum(1 for x in ter_manifest if x["file_name"]),
      "| failed:", sum(1 for x in ter_manifest if x["file_name"] is None))

skip amfi_ter_2023-01.csv (2,565 bytes)
skip amfi_ter_2023-02.csv (2,574 bytes)
skip amfi_ter_2023-03.csv (2,574 bytes)
skip amfi_ter_2023-04.csv (2,535 bytes)
skip amfi_ter_2023-05.csv (2,570 bytes)
skip amfi_ter_2023-06.csv (2,603 bytes)
skip amfi_ter_2023-07.csv (15,795 bytes)
skip amfi_ter_2023-08.csv (16,521 bytes)
skip amfi_ter_2023-09.csv (2,658 bytes)
skip amfi_ter_2023-10.csv (2,677 bytes)
skip amfi_ter_2023-11.csv (2,677 bytes)
skip amfi_ter_2023-12.csv (2,677 bytes)
skip amfi_ter_2024-01.csv (2,677 bytes)
skip amfi_ter_2024-02.csv (2,677 bytes)
skip amfi_ter_2024-03.csv (2,677 bytes)
skip amfi_ter_2024-04.csv (2,676 bytes)
skip amfi_ter_2024-05.csv (2,677 bytes)
skip amfi_ter_2024-06.csv (2,837 bytes)
skip amfi_ter_2024-07.csv (2,979 bytes)
skip amfi_ter_2024-08.csv (2,676 bytes)
skip amfi_ter_2024-09.csv (15,558 bytes)
skip amfi_ter_2024-10.csv (287,718 bytes)
skip amfi_ter_2024-11.csv (282,934 bytes)
skip amfi_ter_2024-12.csv (289,836 bytes)
skip amfi_ter_2025-01.csv (290,

In [10]:
nav_frames = []
nav_errors = []
nav_stub_files = []

for f in sorted(glob.glob(str(RAW / "amfi_nav_*.txt"))):
    if "probe" in f:
        continue
    try:
        d = pd.read_csv(f, sep=";", on_bad_lines="skip", low_memory=False)
        if d.shape[1] < 8:                     # stub file (5, 2) from 2023-01..03
            nav_stub_files.append(Path(f).name)
            print(f"stub  {Path(f).name}  shape={d.shape} — excluded")
            continue
        d["_source_file"] = Path(f).name
        nav_frames.append(d)
        print(f"ok    {Path(f).name}  {d.shape}")
    except Exception as e:
        nav_errors.append((Path(f).name, type(e).__name__, str(e)[:120]))
        print(f"FAIL  {Path(f).name}  {type(e).__name__}: {str(e)[:120]}")

nav_all = pd.concat(nav_frames, ignore_index=True)
print("\nNAV combined shape:", nav_all.shape)
print("files ok:", len(nav_frames),
      "| stubs excluded:", len(nav_stub_files),
      "| failures:", len(nav_errors))
if nav_stub_files:
    print("stub files:", nav_stub_files)
if nav_errors:
    print("failed files:")
    for name, exc, msg in nav_errors:
        print(f"  {name}: {exc}: {msg}")

stub  amfi_nav_2023-01.txt  shape=(5, 1) — excluded
stub  amfi_nav_2023-02.txt  shape=(5, 1) — excluded
stub  amfi_nav_2023-03.txt  shape=(5, 1) — excluded
ok    amfi_nav_2023-04.txt  (134186, 9)
ok    amfi_nav_2023-05.txt  (163365, 9)
ok    amfi_nav_2023-06.txt  (158552, 9)
ok    amfi_nav_2023-07.txt  (159315, 9)
ok    amfi_nav_2023-08.txt  (162418, 9)
ok    amfi_nav_2023-09.txt  (146140, 9)
ok    amfi_nav_2023-10.txt  (152997, 9)
ok    amfi_nav_2023-11.txt  (152803, 9)
ok    amfi_nav_2023-12.txt  (154387, 9)
ok    amfi_nav_2024-01.txt  (161849, 9)
ok    amfi_nav_2024-02.txt  (158807, 9)
ok    amfi_nav_2024-03.txt  (151138, 9)
ok    amfi_nav_2024-04.txt  (151871, 9)
ok    amfi_nav_2024-05.txt  (162868, 9)
ok    amfi_nav_2024-06.txt  (153682, 9)
ok    amfi_nav_2024-07.txt  (174027, 9)
ok    amfi_nav_2024-08.txt  (169298, 9)
ok    amfi_nav_2024-09.txt  (165576, 9)
ok    amfi_nav_2024-10.txt  (177791, 9)
ok    amfi_nav_2024-11.txt  (151064, 9)
ok    amfi_nav_2024-12.txt  (173809, 9)
ok  

In [11]:
small_navs = [f for f in sorted(glob.glob(str(RAW / "amfi_nav_*.txt")))
              if "probe" not in f and Path(f).stat().st_size < 1_000_000]
print("small NAV files (< 1 MB):")
for f in small_navs:
    print(f"  {Path(f).name}  {Path(f).stat().st_size:,} bytes")
    head = Path(f).read_text(encoding="utf-8", errors="ignore")[:200]
    print(f"    first 200 chars: {head!r}")

small NAV files (< 1 MB):
  amfi_nav_2023-01.txt  220,500 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'
  amfi_nav_2023-02.txt  220,500 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'
  amfi_nav_2023-03.txt  220,500 bytes
    first 200 chars: '<!DOCTYPE html><html lang="en" data-theme="light"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="stylesheet" href="/_next/static/css/d0eed'


In [4]:
def read_ter_csv(path):
    d = pd.read_csv(path, on_bad_lines="skip")
    d = d[d["NSDL Scheme Code"].notna()].copy()
    d["TER Date"] = pd.to_datetime(d["TER Date"], errors="coerce")
    d = d[d["TER Date"].notna()].copy()
    d["_source_file"] = Path(path).name
    m = re.search(r"amfi_ter_(\d{4})-(\d{2})", Path(path).name)
    if m:
        d["Month"] = f"{m.group(2)}-{m.group(1)}"
    return d

ter_frames = []
ter_dropped = []

for f in sorted(glob.glob(str(RAW / "amfi_ter_*.csv"))):
    if "probe" in f:
        continue
    raw_n = sum(1 for _ in open(f, encoding="utf-8")) - 1
    d = read_ter_csv(f)
    kept = len(d)
    dropped = max(0, raw_n - kept)
    if dropped:
        ter_dropped.append((Path(f).name, raw_n, kept, dropped))
    ter_frames.append(d)

ter_all = pd.concat(ter_frames, ignore_index=True)

# drop trailing unnamed columns created by on_bad_lines="skip" reading disclaimer rows
ter_all = ter_all.loc[:, [c for c in ter_all.columns if not str(c).startswith("Unnamed")]]

# proper month ordering
ter_all["MonthPeriod"] = (pd.to_datetime(ter_all["Month"], format="%m-%Y").dt.to_period("M"))

print("TER combined shape:", ter_all.shape)
print("columns:", ter_all.columns.tolist())
print("unique NSDL Scheme Code:", ter_all["NSDL Scheme Code"].nunique())
print("unique Month:", ter_all["Month"].nunique())
print("rows dropped as disclaimer/footer:", sum(x[3] for x in ter_dropped))

print("\nTER schemes per month (chronological):")
per_month = (ter_all.groupby("MonthPeriod")["NSDL Scheme Code"]
                    .nunique()
                    .rename_axis("Month")
                    .reset_index())
per_month["Month"] = per_month["Month"].dt.strftime("%m-%Y")
print(per_month.to_string(index=False))

TER combined shape: (47512, 26)
columns: ['NSDL Scheme Code', 'Scheme Name', 'Scheme Type', 'Scheme Category', 'TER Date', 'Regular Plan - Base TER (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Regular Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Regular Plan - GST (%)', 'Regular Plan - Total TER (%)', 'Direct Plan - Base TER (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(b) (%)', 'Direct Plan - Additional expense as per Regulation 52(6A)(c) (%)', 'Direct Plan - GST (%)', 'Direct Plan - Total TER (%)', '_source_file', 'Month', 'Regular Plan - Base Expense Ratio (BER) (%)', 'Regular Plan - Brokerage cost (%)', 'Regular Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Regular Plan - Statutory Levies (including GST) (%)', 'Direct Plan - Base Expense Ratio (BER) (%)', 'Direct Plan - Brokerage cost (%)', 'Direct Plan - Transaction Cost incurred for the purpose of execution of trade (%)', 'Direct Pla

In [ ]:
nav_all["Date"] = pd.to_datetime(nav_all["Date"], format="%d-%b-%Y", errors="coerce")
nav_all["MonthPeriod"] = nav_all["Date"].dt.to_period("M")

def is_large_cap(name):
    if not isinstance(name, str):
        return False
    s = " ".join(name.lower().split())
    return ("large cap" in s or "largecap" in s) and ("mid" not in s)

nav_all["is_large_cap_name"] = nav_all["NAV Name"].apply(is_large_cap)
ter_all["is_large_cap_cat"] = ter_all["Scheme Category"].str.contains(
    "Large Cap", case=False, na=False)

USABLE_START = pd.Period("2024-10", freq="M")
nav_usable = nav_all[nav_all["MonthPeriod"] >= USABLE_START].copy()
ter_usable = ter_all[ter_all["MonthPeriod"] >= USABLE_START].copy()

print("usable window:", USABLE_START, "→ 2026-09")
print("NAV usable shape:", nav_usable.shape)
print("TER usable shape:", ter_usable.shape)
print()
print("NAV large-cap rows:", nav_usable["is_large_cap_name"].sum(), "of", len(nav_usable))
print("TER large-cap rows:", ter_usable["is_large_cap_cat"].sum(), "of", len(ter_usable))
print("NAV large-cap unique names:",
      nav_usable.loc[nav_usable["is_large_cap_name"], "NAV Name"].nunique())
print("TER large-cap unique schemes:",
      ter_usable.loc[ter_usable["is_large_cap_cat"], "NSDL Scheme Code"].nunique())

print("\nlarge-cap label variants:")
print(ter_usable.loc[ter_usable["is_large_cap_cat"], "Scheme Category"].value_counts().to_string())

print("\nTER large-cap schemes per month (usable window):")
print(ter_usable[ter_usable["is_large_cap_cat"]]
      .groupby("MonthPeriod")["NSDL Scheme Code"]
      .nunique().sort_index().to_string())

In [ ]:
for df in (nav_usable, nav_all):
    df["Net Asset Value"] = pd.to_numeric(df["Net Asset Value"], errors="coerce")
    if "MonthPeriod" in df.columns:
        df["MonthStr"] = df["MonthPeriod"].astype(str)
        df = df.drop(columns=["MonthPeriod"], errors="ignore")

nav_usable["Net Asset Value"] = pd.to_numeric(nav_usable["Net Asset Value"], errors="coerce")
nav_all["Net Asset Value"]    = pd.to_numeric(nav_all["Net Asset Value"], errors="coerce")

nav_usable["MonthStr"] = nav_usable["MonthPeriod"].astype(str)
nav_all["MonthStr"]    = nav_all["MonthPeriod"].astype(str)

nav_usable.drop(columns=["MonthPeriod"], errors="ignore", inplace=True)
nav_all.drop(columns=["MonthPeriod"], errors="ignore", inplace=True)

ter_usable["MonthStr"] = ter_usable["MonthPeriod"].astype(str)
ter_all["MonthStr"]    = ter_all["MonthPeriod"].astype(str)

ter_usable.drop(columns=["MonthPeriod"], errors="ignore", inplace=True)
ter_all.drop(columns=["MonthPeriod"], errors="ignore", inplace=True)

# write
nav_usable.to_parquet(INTERIM / "nav_usable_2024-10_2026-09.parquet", index=False)
ter_usable.to_parquet(INTERIM / "ter_usable_2024-10_2026-09.parquet", index=False)
print("wrote nav_usable:", len(nav_usable), "rows")
print("wrote ter_usable:", len(ter_usable), "rows")

nav_all.to_parquet(INTERIM / "nav_all_2023-04_2026-09.parquet", index=False)
ter_all.to_parquet(INTERIM / "ter_all_2023-07_2026-09.parquet", index=False)
print("wrote full combined parquet:", len(nav_all), "nav rows,", len(ter_all), "ter rows")

In [7]:
nav_check = pd.read_parquet(INTERIM / "nav_usable_2024-10_2026-09.parquet")
ter_check = pd.read_parquet(INTERIM / "ter_usable_2024-10_2026-09.parquet")
print("nav_usable:", nav_check.shape)
print(nav_check.dtypes)
print("\nter_usable:", ter_check.shape)
print(ter_check.dtypes)

nav_usable: (4255618, 11)
Scheme Code                               str
NAV Name                                  str
Plan                                      str
Option                                    str
ISIN Div Payout/ISIN Growth               str
ISIN Div Reinvestment                     str
Net Asset Value                       float64
Date                           datetime64[us]
_source_file                              str
is_large_cap_name                        bool
MonthStr                                  str
dtype: object

ter_usable: (47228, 27)
NSDL Scheme Code                                                                                 str
Scheme Name                                                                                      str
Scheme Type                                                                                      str
Scheme Category                                                                                  str
TER Date                 